In [1]:
# base
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic() 
model = "claude-haiku-4-5"

In [2]:
# Helpers
def add_user_message(messages, text):
  user_message = {
    "role": "user", 
    "content": text
  }
  messages.append(user_message)

def add_assistant_message(messages, text):
  assistant_message = {
    "role": "assistant",
    "content": text
  }
  messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
  params= {
    "model": model,
    "max_tokens": 1000,
    "messages": messages,
    "temperature": temperature,
    "stop_sequences": stop_sequences
  }

  if system:
    params["system"] = system

  message = client.messages.create(**params)
  return message.content[0].text

In [3]:
# Function to generate a new dataset
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [4]:
# generate dataset
# dataset = generate_dataset()
# with open("dataset.json", "w") as f:
#   json.dump(dataset, f, indent=2)

In [27]:
def run_prompt(test_case):
  """Merges the prompt and test case input, then returns the result"""
  prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation
"""
  messages = []
  add_user_message(messages, prompt)
  # uses ```code if unknown whether it'd returns ```python, ```bash, ```json, ```regex, etc`
  add_assistant_message(messages, "```code")
  output = chat(
    messages,
    stop_sequences=["```"]
  )
  return output

In [28]:
# grade_by_model from text based guide, high possibility of returning error/wrong since it's using like completely different variables
# replaced previously {task} and {solution} to {test_case} and {output} accordingly. Might need to test this function too
def grade_by_model_0(test_case, output):
  eval_prompt = f"""
You are an expert code reviewer. Evaluate this AI-generated solution.

Task: {test_case["task"]}
Solution: {output}

Provide your evaluation as a structured JSON object with:
- "strenghts": An array of 1-3 key strength
- "weaknesses": An array of 1-3 key areas of improvements
- "reasoning" "A concise explanation of your assessment
- "score": A number between 1-10
"""

  messages = []
  add_user_message(messages, eval_prompt)
  add_assistant_message(messages, "```json")

  eval_text = chat(messages, stop_sequences=["```"])
  return json.loads(eval_text)

In [29]:
# Model Validator - grading by using models (basically using AI to score them results)
def grade_by_model(test_case, output):
  eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strenghts
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
  "strengths": string[],
  "weaknesses": string[],
  "reasoning": string,
  "score": number
}}
"""
  
  messages = []
  add_user_message(messages, eval_prompt)
  add_assistant_message(messages, "```json")
  eval_text = chat(messages, stop_sequences=["```"])
  return json.loads(eval_text)

In [30]:
# code validators

import ast
import re

def validate_json(text):
  try:
    json.loads((text.strip()))
    return 10
  except json.JSONDecodeError:
    return 0
  
def validate_python(text):
  try:
    ast.parse(text.strip())
    return 10
  except SyntaxError:
    return 0
  
def validate_regex(text):
  try:
    re.compile(text.strip())
    return 10
  except re.error:
    return 0
  
def grade_syntax(response, test_case):
  format = test_case["format"]

  match format:
    case "json":
      return validate_json(response)
    case "python":
      return validate_python(response)
    case "regex":
      return validate_regex(response)
    case _:
      return 0 # because of it's unkown format
  

In [31]:
def run_test_case(test_case):
  """Calls run_prompt, then grades the result"""
  output = run_prompt(test_case)
  
  model_grade = grade_by_model(test_case, output)
  model_score = model_grade["score"]
  model_reasoning = model_grade["reasoning"]
  syntax_score = grade_syntax(output, test_case)

  score = (model_score + syntax_score)/2

  return {
    "output": output,
    "test_case": test_case,
    # "model_score": model_grade["score"],
    # "reasoning": model_grade["reasoning"],
    # can skip strength & weakness to avoid being too long
    # "strength": model_grade["strengths"],
    # "weaknesses": model_grade["weaknesses"]
    # "model_score": model_score,
    "score": score,
    "reasoning": model_reasoning,
    # "syntax_score": syntax_score
  }



In [32]:
from statistics import mean

def run_eval(dataset):
  """Loads the dataset and calls run_test_case with each case"""
  results = []

  for test_case in dataset:
    result = run_test_case(test_case)
    results.append(result)

  average_score = mean([result["score"] for result in results])
  print(f"Average score: {average_score}")


  return results

In [33]:
with open("dataset.json", "r") as f:
  dataset = json.load(f)

results = run_eval(dataset)

Average score: 7.666666666666667


In [35]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\nimport re\n\ndef extract_aws_account_id(arn):\n    match = re.search(r'arn:aws:[^:]+:([^:]*):([^:]+):', arn)\n    if match:\n        return match.group(2) if match.group(2) else None\n    return None\n\n# Test\nprint(extract_aws_account_id('arn:aws:s3:::my-bucket/key'))\nprint(extract_aws_account_id('arn:aws:iam::123456789012:user/Development/product_1234/*'))\n",
    "test_case": {
      "task": "Extract the AWS account ID from an ARN string like 'arn:aws:s3:::my-bucket/key'",
      "format": "regex"
    },
    "score": 8.0,
    "reasoning": "The solution correctly extracts account IDs from well-formed ARNs and properly handles the S3 case where the account ID field is empty. However, it lacks robustness through input validation and error handling. The function would fail silently or raise unhelpful exceptions on edge cases like non-string inputs or severely malformed ARNs. A production solution should validate inputs and provide clearer error feedback."
  },
  